# Does RL teach new skills or only sharpen? Step 1: small-scale reproduction

Reproduces the core experiment of **Yue et al. (2025), "Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base Model?"** on a model small enough for a free GPU.

**Setup:** Qwen2.5-0.5B base, GSM8K, GRPO with a LoRA adapter (Unsloth + TRL), sampling with vLLM.

**Three models are compared:**
1. `base`: the untouched base model
2. `rl`: GRPO with a correctness reward
3. `random`: GRPO with a coin-flip reward (control for contamination and format effects)

**Measurements:** pass@k for k = 1 to 256, solved-set overlap, answer-format rate, and perplexity of each model's outputs under the base model.

## How to run
- Runtime: **GPU (T4)** on Colab, or a Kaggle GPU notebook.
- Each stage saves to Google Drive (Colab) or `/kaggle/working` (Kaggle) and skips work already done, so a disconnect never loses progress.
- **Restart the runtime between GPU stages** (vLLM and training each hold GPU memory). After every restart, run the **Setup** cell again, then the next stage.
- Rough cost on a T4: Stage A ~45 min, Stage B ~1.5 to 2.5 h per run, Stage C ~45 min per model. Lower `N_EVAL` or `GRPO_STEPS` for a quick end-to-end test first.

In [ ]:
# Run once per fresh runtime.
# If pip reports version conflicts, copy the install cell from Unsloth's current
# GRPO Colab notebook instead: their pinned versions change often.
%pip install -q unsloth vllm
%pip install -q datasets matplotlib

## Setup (run after every restart)
Config, storage, data, prompt, grading and pass@k. For Step 2, change `BASE_MODEL` and `RUN_NAME` (e.g. to the Instruct checkpoint) and rerun everything.

In [ ]:
import os, re, json, math, random, gc
import numpy as np

# ---------------- Config ----------------
BASE_MODEL  = "Qwen/Qwen2.5-0.5B"     # Step 2: also run "Qwen/Qwen2.5-0.5B-Instruct"
RUN_NAME    = "qwen2.5-0.5b-base"     # results folder; change it when you change the model
N_EVAL      = 200                     # GSM8K test problems
N_SAMPLES   = 256                     # samples per problem, so k goes up to 256
MAX_TOKENS  = 512
TEMPERATURE = 0.6                     # eval sampling (as in Yue et al.)
TOP_P       = 0.95
SEED        = 0
LORA_RANK   = 32
GRPO_STEPS  = 300
CHUNK       = 25                      # eval problems per saved sampling chunk

# ---------------- Storage ----------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/rl_sharpening"
except Exception:
    ROOT = "/kaggle/working/rl_sharpening" if os.path.exists("/kaggle") else "./rl_sharpening"
OUT = os.path.join(ROOT, RUN_NAME)
os.makedirs(OUT, exist_ok=True)
print("Saving to", OUT)

def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as f:
        json.dump(obj, f)

def load_json(name):
    with open(os.path.join(OUT, name)) as f:
        return json.load(f)

def exists(name):
    return os.path.exists(os.path.join(OUT, name))

# ---------------- Data and prompt ----------------
from datasets import load_dataset

gsm = load_dataset("openai/gsm8k", "main")

def gold_number(answer_field):
    return answer_field.split("####")[-1].strip().replace(",", "")

def fmt_solution(ans):
    """GSM8K solution without calculator annotations, ending in '#### N'."""
    body, final = ans.split("####")
    body = re.sub(r"<<[^>]*>>", "", body).strip()
    return f"{body}\n#### {final.strip()}"

# Two worked examples teach the base model the '#### N' answer format.
# Every model (base, RL, Instruct) gets the identical raw prompt, so comparisons are controlled.
FEWSHOT = gsm["train"].select([0, 1])
PREFIX = "".join(f"Question: {ex['question']}\nAnswer: {fmt_solution(ex['answer'])}\n\n" for ex in FEWSHOT)

def make_prompt(question):
    return PREFIX + f"Question: {question}\nAnswer:"

eval_set = gsm["test"].shuffle(seed=SEED).select(range(N_EVAL))
EVAL_PROMPTS = [make_prompt(q) for q in eval_set["question"]]
EVAL_GOLD = [gold_number(a) for a in eval_set["answer"]]
train_set = gsm["train"].select(range(2, len(gsm["train"])))   # skip the few-shot examples

# ---------------- Grading ----------------
NUM = r"-?\d[\d,]*\.?\d*"

def first_answer(text):
    """Keep only the first answer; base models often go on to write new questions."""
    return text.split("Question:")[0]

def extract(text):
    """Strict extraction: '#### N' or '\\boxed{N}'. No last-number fallback, which would
    let random numbers count as correct at large k."""
    text = first_answer(text)
    m = re.search(r"####\s*\$?\s*(" + NUM + ")", text)
    if m is None:
        m = re.search(r"\\boxed\{\s*\$?\s*(" + NUM + ")", text)
    return m.group(1).replace(",", "").rstrip(".") if m else None

def is_correct(text, gold):
    pred = extract(text)
    if pred is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False

assert is_correct(" She has 3 + 4 = 7 apples.\n#### 7", "7")
assert is_correct(" ... so \\boxed{1,200}", "1200")
assert not is_correct(" The answer is 7", "7")

# ---------------- pass@k ----------------
KS = [1, 2, 4, 8, 16, 32, 64, 128, 256]

def pass_at_k(n, c, k):
    """Unbiased estimator (Chen et al. 2021): 1 - C(n-c, k) / C(n, k)."""
    if n - c < k:
        return 1.0
    return float(1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))

assert pass_at_k(256, 0, 256) == 0.0 and pass_at_k(256, 1, 256) == 1.0
assert abs(pass_at_k(10, 5, 1) - 0.5) < 1e-9

# ---------------- Sampling with vLLM ----------------
def sample_all(tag, lora_path=None):
    """N_SAMPLES completions per eval problem. Saved per chunk, so a rerun resumes."""
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    llm = LLM(model=BASE_MODEL, dtype="half", gpu_memory_utilization=0.85, seed=SEED,
              max_model_len=2048, enable_lora=lora_path is not None, max_lora_rank=LORA_RANK)
    sp = SamplingParams(n=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P,
                        max_tokens=MAX_TOKENS, stop=["\nQuestion:"], seed=SEED)
    lora = LoRARequest(tag, 1, lora_path) if lora_path else None
    for start in range(0, N_EVAL, CHUNK):
        name = f"samples_{tag}_{start:04d}.json"
        if exists(name):
            continue
        outs = llm.generate(EVAL_PROMPTS[start:start + CHUNK], sp, lora_request=lora)
        save_json([[o.text for o in out.outputs] for out in outs], name)
        print(f"{tag}: {min(start + CHUNK, N_EVAL)}/{N_EVAL} problems sampled")
    del llm
    gc.collect()

def load_samples(tag):
    texts = []
    for start in range(0, N_EVAL, CHUNK):
        texts += load_json(f"samples_{tag}_{start:04d}.json")
    return texts

print("Setup done.", len(EVAL_PROMPTS), "eval problems,", len(train_set), "train problems")

## Stage A: sample the base model
256 samples for each of the 200 eval problems. Restart the runtime when it finishes.

In [ ]:
sample_all("base")

## Stage B: GRPO training
Run once with `REWARD = "correct"` (tag `rl`) and once with `REWARD = "random"` (tag `random`), restarting the runtime between runs.

- The reward is binary correctness with the same strict grader used in evaluation.
- The random control pays 1 with probability 0.5 regardless of the answer. If it also improves pass@1, part of the "RL gain" is not learning from the task (format, or contamination).
- `beta=0.0` (no KL penalty) follows most recent RLVR work; try `0.04` as an ablation.

In [ ]:
REWARD = "correct"          # "correct" -> tag "rl", "random" -> tag "random"
TAG = "rl" if REWARD == "correct" else "random"

from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=False,
    fast_inference=True, max_lora_rank=LORA_RANK, gpu_memory_utilization=0.6)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

train_ds = train_set.map(
    lambda ex: {"prompt": make_prompt(ex["question"]), "gold": gold_number(ex["answer"])},
    remove_columns=train_set.column_names).shuffle(seed=SEED)

coin = random.Random(SEED)

def reward_fn(prompts, completions, gold, **kwargs):
    if REWARD == "random":
        return [float(coin.random() < 0.5) for _ in completions]
    return [1.0 if is_correct(c, g) else 0.0 for c, g in zip(completions, gold)]

ckpt_dir = os.path.join(OUT, f"ckpt_{TAG}")
config = GRPOConfig(
    output_dir=ckpt_dir,
    learning_rate=1e-5,                 # LoRA tolerates a higher LR than full fine-tuning
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    beta=0.0,
    temperature=1.0,                    # training rollouts; eval uses TEMPERATURE above
    num_generations=8,                  # group size for GRPO advantages
    per_device_train_batch_size=8,      # completions per device step (= 1 prompt x 8)
    gradient_accumulation_steps=4,      # 4 prompts per optimizer step
    max_prompt_length=768,
    max_completion_length=384,
    max_steps=GRPO_STEPS,
    logging_steps=5,
    save_steps=50,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    report_to="none",
    seed=SEED,
)

trainer = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=[reward_fn],
                      args=config, train_dataset=train_ds)
has_ckpt = os.path.isdir(ckpt_dir) and any(d.startswith("checkpoint-") for d in os.listdir(ckpt_dir))
trainer.train(resume_from_checkpoint=has_ckpt)

model.save_pretrained(os.path.join(OUT, f"lora_{TAG}"))
tokenizer.save_pretrained(os.path.join(OUT, f"lora_{TAG}"))
save_json(trainer.state.log_history, f"train_log_{TAG}.json")
print("Saved adapter for", TAG)

In [ ]:
# Training curve: the correctness reward should rise; the random reward should hover near 0.5.
import matplotlib.pyplot as plt
for tag in ["rl", "random"]:
    if exists(f"train_log_{tag}.json"):
        log = [r for r in load_json(f"train_log_{tag}.json") if "reward" in r]
        plt.plot([r["step"] for r in log], [r["reward"] for r in log], label=tag)
plt.xlabel("step"); plt.ylabel("mean reward"); plt.legend(); plt.title("GRPO reward"); plt.show()

## Stage C: sample the trained models
Same prompts and sampling settings as Stage A, with the LoRA adapter loaded in vLLM. Restart between the two calls if memory runs short.

In [ ]:
for tag in ["rl", "random"]:
    path = os.path.join(OUT, f"lora_{tag}")
    if os.path.isdir(path):
        sample_all(tag, lora_path=path)
    else:
        print("No adapter yet for", tag)

## Stage D: analysis (CPU is fine)
1. **pass@k curves.** Does the base model cross above RL at large k?
2. **Sampling efficiency gap** = RL pass@1 minus base pass@256 (Yue et al.'s measure of headroom).
3. **Solved-set overlap.** Problems RL solves that the base never solves in 256 tries are the evidence for a new skill.
4. **Format rate.** Share of samples with an extractable answer. A big RL jump here means some gain is just format.
5. **Buckets by base solve rate.** Preview of the Step 2 analysis: where does RL change things?

In [ ]:
import matplotlib.pyplot as plt

TAGS = [t for t in ["base", "rl", "random"] if exists(f"samples_{t}_0000.json")]

def score(tag):
    texts = load_samples(tag)
    correct = np.array([[is_correct(t, g) for t in row] for row, g in zip(texts, EVAL_GOLD)])
    fmt = np.mean([[extract(t) is not None for t in row] for row in texts])
    return correct, float(fmt)

correct, fmt_rate = {}, {}
for t in TAGS:
    correct[t], fmt_rate[t] = score(t)

curves = {t: [float(np.mean([pass_at_k(N_SAMPLES, int(c), k) for c in correct[t].sum(1)])) for k in KS]
          for t in TAGS}

print(f"{'k':>5}" + "".join(f"{t:>10}" for t in TAGS))
for i, k in enumerate(KS):
    print(f"{k:>5}" + "".join(f"{curves[t][i]:>10.3f}" for t in TAGS))
print("\nformat rate:", {t: round(v, 3) for t, v in fmt_rate.items()})

plt.figure(figsize=(6, 4))
for t in TAGS:
    plt.plot(KS, curves[t], marker="o", label=t)
plt.xscale("log", base=2); plt.xticks(KS, KS)
plt.xlabel("k"); plt.ylabel("pass@k"); plt.title(f"GSM8K pass@k, {BASE_MODEL}")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT, "pass_at_k.png"), dpi=200); plt.show()

In [ ]:
summary = {"config": {"model": BASE_MODEL, "n_eval": N_EVAL, "n_samples": N_SAMPLES,
                      "temperature": TEMPERATURE, "grpo_steps": GRPO_STEPS},
           "ks": KS, "pass_at_k": curves, "format_rate": fmt_rate, "overlap": {}, "buckets": {}}

solved = {t: correct[t].any(1) for t in TAGS}
for t in TAGS:
    if t == "base":
        continue
    gap = curves[t][0] - curves["base"][-1]
    ov = {"both": int((solved[t] & solved["base"]).sum()),
          "new_only": int((solved[t] & ~solved["base"]).sum()),
          "lost": int((~solved[t] & solved["base"]).sum()),
          "neither": int((~solved[t] & ~solved["base"]).sum()),
          "new_only_ids": [int(i) for i in np.where(solved[t] & ~solved["base"])[0]],
          "sampling_efficiency_gap": float(gap)}
    summary["overlap"][t] = ov
    print(f"{t}: solved by both {ov['both']}, only {t} {ov['new_only']}, lost {ov['lost']}, "
          f"neither {ov['neither']}; efficiency gap {gap:+.3f}")

# Buckets by the base model's per-problem solve rate (Step 2 preview)
base_rate = correct["base"].mean(1)
buckets = {"never (0%)": base_rate == 0,
           "rare (<5%)": (base_rate > 0) & (base_rate < 0.05),
           "sometimes (5-50%)": (base_rate >= 0.05) & (base_rate < 0.5),
           "often (>=50%)": base_rate >= 0.5}
print(f"\n{'bucket':<20}{'n':>5}" + "".join(f"{t + ' p@1':>13}{t + ' p@256':>14}" for t in TAGS))
for name, mask in buckets.items():
    row = {}
    for t in TAGS:
        c = correct[t][mask].sum(1)
        p1 = float(np.mean([pass_at_k(N_SAMPLES, int(x), 1) for x in c])) if mask.any() else float("nan")
        p256 = float(np.mean([pass_at_k(N_SAMPLES, int(x), 256) for x in c])) if mask.any() else float("nan")
        row[t] = {"pass@1": p1, "pass@256": p256}
    summary["buckets"][name] = {"n": int(mask.sum()), **row}
    print(f"{name:<20}{int(mask.sum()):>5}" + "".join(f"{row[t]['pass@1']:>13.3f}{row[t]['pass@256']:>14.3f}" for t in TAGS))

save_json(summary, "results.json")

**Check before you trust "new_only".** At k = 256 a model can occasionally land on the right number by luck. Read a few correct RL outputs on the `new_only_ids` problems below and confirm the reasoning is actually valid before calling it a new skill.

In [ ]:
if "rl" in TAGS:
    texts = load_samples("rl")
    for i in summary["overlap"]["rl"]["new_only_ids"][:3]:
        good = [t for t in texts[i] if is_correct(t, EVAL_GOLD[i])]
        print(f"--- problem {i}: {len(good)}/{N_SAMPLES} correct (base: 0/{N_SAMPLES}) ---")
        print(eval_set[i]["question"])
        print(">>>", first_answer(good[0])[:800], "\n")

## Stage E: perplexity under the base model (GPU)
Yue et al.'s second test: if RL outputs are **less** surprising to the base model than the base model's own samples, RL is picking paths the base already preferred (sharpening). Higher perplexity would suggest RL moved outside the base distribution.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PPL_PROBLEMS, PPL_PER_PROBLEM = 100, 4

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
lm = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).cuda().eval()

def completion_ppl(prompt, completion):
    p_ids = tok(prompt, return_tensors="pt").input_ids
    c_ids = tok(completion, return_tensors="pt", add_special_tokens=False).input_ids
    if c_ids.shape[1] == 0:
        return None
    ids = torch.cat([p_ids, c_ids], dim=1).cuda()
    with torch.no_grad():
        logits = lm(ids).logits[0, p_ids.shape[1] - 1:-1].float()
    nll = torch.nn.functional.cross_entropy(logits, ids[0, p_ids.shape[1]:])
    return math.exp(nll.item())

ppl = {}
for tag in [t for t in ["base", "rl", "random"] if exists(f"samples_{t}_0000.json")]:
    rng, texts, vals = random.Random(SEED), load_samples(tag), []
    for i in range(min(PPL_PROBLEMS, N_EVAL)):
        for t in rng.sample(texts[i], PPL_PER_PROBLEM):
            v = completion_ppl(EVAL_PROMPTS[i], first_answer(t))
            if v is not None:
                vals.append(v)
    ppl[tag] = {"median": float(np.median(vals)), "mean": float(np.mean(vals)), "n": len(vals)}
    print(f"{tag:>7}: median ppl under base {ppl[tag]['median']:.3f} (n={len(vals)})")

summary = load_json("results.json")
summary["perplexity_under_base"] = ppl
save_json(summary, "results.json")

## Reading the results
- **Sharpening (Yue et al. replicates):** RL beats base at pass@1, base catches up or wins by pass@256, `new_only` is near zero, and RL perplexity under base is at or below base's own.
- **Expansion:** RL stays above base at pass@256 and `new_only` holds problems with valid reasoning.
- **Random reward also helps:** some of the gain is format or contamination, not task learning. Compare the format rates.

Whatever you find, publish it with the plot, `results.json`, and this notebook. Then run Step 2: set `BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"` and `RUN_NAME = "qwen2.5-0.5b-instruct"`, rerun every stage, and compare the two runs' bucket tables.